# Generative Language Models

## Project 2 - Fine-Tune Longformer for ArXiv Paper Classification

---

[Longformer Base 4096] (https://huggingface.co/allenai/longformer-base-4096)

[Arxiv dataset] (https://huggingface.co/datasets/nick007x/arxiv-papers)

[Direct link to Arxiv dataset .parquet file (metadata that includes the abstract] (https://huggingface.co/datasets/nick007x/arxiv-papers/resolve/main/train.parquet)

## 1. Import Dependencies

---

In [ ]:
import os
import json
from datetime import datetime
import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from peft import LoraConfig, PeftModel, get_peft_model

from data_collator_with_padding import DataCollatorWithGlobalPadding

In [ ]:
torch.set_float32_matmul_precision("high")

In [ ]:
# Quiet the tokenizer fork warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Model Configurations

---

In [ ]:
MODEL_CONFIGS = {
    "longformer-base": {
        "name": "models/longformer-base-4096",
        "max_length": 1024,
        "batch_size": 12,
        "gradient_accumulation": 4,
        "needs_global_attention": False,
    },
    "longformer-large": {
        "name": "models/longformer-large-4096",
        "max_length": 4096,
        "batch_size": 8,
        "gradient_accumulation": 4,
        "needs_global_attention": False,
    },
    "led-base": {
        "name": "models/led-base-16384",
        "max_length": 8192,
        "batch_size": 8,
        "gradient_accumulation": 4,
        "needs_global_attention": True,
    },
    "led-large": {
        "name": "models/led-large-16384",
        "max_length": 8192,
        "batch_size": 4,
        "gradient_accumulation": 8,
        "needs_global_attention": True,
    },
}

# Selected model
SELECTED_MODEL = "longformer-base"  

## 3. Data Loading

---

In [ ]:
# Path to parquet file
PARQUET_FILE_PATH = "./data/arxiv_papers/train.parquet"

# Get selected configuration
config = MODEL_CONFIGS[SELECTED_MODEL]
model_name = config["name"]
max_length = config["max_length"]
batch_size = config["batch_size"]
gradient_accumulation_steps = config["gradient_accumulation"]
needs_global_attention = config["needs_global_attention"]

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Model: {model_name}")
print(f"Max Length: {max_length}")
print(f"Batch Size: {batch_size}")
print(f"Gradient Accumulation: {gradient_accumulation_steps}")
print(f"Effective Batch Size: {batch_size * gradient_accumulation_steps}")
print("=" * 60)

# Create output directory with timestamp and model name
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"./results_{SELECTED_MODEL}_{timestamp}"
os.makedirs(output_dir, exist_ok=True)

print("\nLoading dataset from local parquet file...")

if not os.path.exists(PARQUET_FILE_PATH):
    print(f"Error: Parquet file not found at {PARQUET_FILE_PATH}")
    print("Please update PARQUET_FILE_PATH in the script.")
    exit(1)

# Load the parquet file
df = pd.read_parquet(PARQUET_FILE_PATH)
print(f"Loaded {len(df):,} total papers")

# Check required columns
required_columns = ['abstract', 'primary_subject']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"Error: Missing required columns: {missing_columns}")
    print(f"Available columns: {list(df.columns)}")
    exit(1)

print(f"✓ Found required columns")

# Remove rows with missing abstracts
original_len = len(df)
df = df.dropna(subset=['abstract', 'primary_subject'])
if len(df) < original_len:
    print(f"Removed {original_len - len(df)} rows with missing data")

# Balanced subset with K per subject
K = 20
df = (df.groupby('primary_subject', group_keys=False)
        .apply(lambda g: g.sample(n=min(K, len(g)), random_state=42))
        .reset_index(drop=True))
print(f"✓ Balanced subset with K={K} → {len(df):,} rows across {df['primary_subject'].nunique()} subjects")


# Show category distribution BEFORE filtering
print(f"\nInitial category distribution:")
category_counts = df['primary_subject'].value_counts()
print(f"Total categories: {len(category_counts)}")
print(f"\nTop 10 categories:")
for cat, count in category_counts.head(10).items():
    print(f"  {cat:50s} {count:>6,}")

# Filter out categories with too few samples (need at least 2 for train/val split)
MIN_SAMPLES_PER_CATEGORY = 2
print(f"\nFiltering categories with less than {MIN_SAMPLES_PER_CATEGORY} samples...")
category_counts = df['primary_subject'].value_counts()
valid_categories = category_counts[category_counts >= MIN_SAMPLES_PER_CATEGORY].index
df = df[df['primary_subject'].isin(valid_categories)]

removed_categories = len(category_counts) - len(valid_categories)
if removed_categories > 0:
    print(f"Removed {removed_categories} categories with too few samples")
    print(f"Remaining papers: {len(df):,}")
    print(f"Remaining categories: {len(valid_categories)}")

# Show final category distribution
category_counts = df['primary_subject'].value_counts()
print(f"\nFinal category distribution (top 10):")
for cat, count in category_counts.head(10).items():
    print(f"  {cat:50s} {count:>6,}")

# Split into train/validation (80/20 split)
print("\nSplitting into train/validation sets...")
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['primary_subject']  # Maintains category distribution
)

print(f"✓ Train samples: {len(train_df):,}")
print(f"✓ Validation samples: {len(val_df):,}")

## 4. Create Label Mappings

---

In [ ]:
print("\nCreating label mapping...")

categories = sorted(df['primary_subject'].unique())
category_to_id = {cat: idx for idx, cat in enumerate(categories)}
id_to_category = {idx: cat for idx, cat in enumerate(categories)}
num_categories = len(categories)

print(f"✓ Total categories: {num_categories}")

# Save category mapping
mapping_file = os.path.join(output_dir, "category_mapping.json")
with open(mapping_file, 'w') as f:
    json.dump({
        'category_to_id': category_to_id,
        'id_to_category': {str(k): v for k, v in id_to_category.items()},
        'num_categories': num_categories
    }, f, indent=2)
print(f"✓ Saved category mapping to {mapping_file}")

# Make labels multi-hot
def add_label(example):
	# one-hot (or later, soft) vector
	vec = np.zeros(num_categories, dtype=np.float32)
	idx = category_to_id[example["primary_subject"]]
	vec[idx] = 1.0
	example["labels"] = vec
	return example

# Convert to HuggingFace Dataset format
print("\nConverting to HuggingFace Dataset format...")
train_dataset = Dataset.from_pandas(train_df, preserve_index=False).map(add_label)
val_dataset = Dataset.from_pandas(val_df,   preserve_index=False).map(add_label)

print(f"✓ Datasets ready")
print(f"  Train columns: {train_dataset.column_names}")

## 5. Load Tokenizer and Model

---

In [ ]:
print(f"\nLoading tokenizer and model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_categories,
    # Change the default loss to BCEWithLogitsLoss, so each class gets
    # its own independent probability instead of competing in a Softmax
    problem_type="multi_label_classification",
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,                 # start with 8–16
    lora_alpha=32,        # 2×r is a good rule
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    # Longformer uses RoBERTa-style names; substring match is fine:
    target_modules=["query", "key", "value", "dense"]  # attn QKV + proj + MLP denselayers
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"Model loaded with {num_categories} output classes")

## 6. Data Pre-Processing

---

In [ ]:
def preprocess_function(batch):
	texts = [
		f"title: {t} {tokenizer.sep_token} abstract: {a}"
		for t, a in zip(batch["title"], batch["abstract"])
	]
	enc = tokenizer(texts, padding=False, truncation=True, max_length=max_length)
	return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

print("\nPreprocessing datasets...")

# Get columns to remove (everything except what we need)
remove_cols_train = [c for c in train_dataset.column_names if c not in ("title","abstract","labels")]
remove_cols_val   = [c for c in val_dataset.column_names   if c not in ("title","abstract","labels")]

train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=batch_size,
    remove_columns=remove_cols_train,
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=batch_size,
    remove_columns=remove_cols_val,
)

# Set format for PyTorch
columns = ["input_ids", "attention_mask", "labels"]
if needs_global_attention:
    columns.append("global_attention_mask")

train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)

print(f"Preprocessing complete")

In [ ]:
# Inspect the labels
item = train_dataset[0]
print(type(item["labels"]), item["labels"].shape if hasattr(item["labels"], "shape") else "no shape")
print(item["labels"])

## 7. Evaluation Metrics

---

In [ ]:
def topk_acc(logits, labels, k=5):
    topk = np.argpartition(logits, -k, axis=1)[:, -k:]
    return float(np.mean([l in row for l, row in zip(labels, topk)]))

def compute_metrics(eval_pred):
	logits, labels = eval_pred
	if isinstance(logits, (tuple, list)):
		logits = logits[0]

	# labels: [N, num_categories] one-hot/soft
	if labels.ndim == 2:
		true_ids = labels.argmax(axis=1)
	else:
		true_ids = labels

	# For predictions, we can just use argmax over logits
	pred_ids = logits.argmax(axis=1)

	acc = float((pred_ids == true_ids).mean())
	prec, rec, f1_w, _ = precision_recall_fscore_support(
		true_ids, pred_ids, average="weighted", zero_division=0
	)
	f1_macro = f1_score(true_ids, pred_ids, average="macro", zero_division=0)
	top5 = topk_acc(logits, true_ids, k=5)

	return {
		"accuracy": acc,
		"f1": float(f1_w),
		"precision": float(prec),
		"recall": float(rec),
		"f1_macro": float(f1_macro),
		"top5": top5,
	}

## 8. Training

---

In [ ]:
# Training arguments
training_args = TrainingArguments(
    label_names=["labels"],
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    eval_accumulation_steps=32,
    num_train_epochs=4,
    logging_steps=100,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    weight_decay=0.01,
    warmup_ratio=0.06,
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    save_total_limit=2,
    optim="adamw_torch_fused",
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    dataloader_num_workers=8,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

# Trainer
data_collator = DataCollatorWithGlobalPadding(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.add_callback(
    EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=1e-3
    )
)

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

start_time = datetime.now()
train_result = trainer.train()
end_time = datetime.now()
# In hours
training_time = (end_time - start_time).total_seconds() / 3600

## 9. Evaluation
---

In [ ]:
print("\n" + "=" * 60)
print("EVALUATING MODEL")
print("=" * 60)

eval_results = trainer.evaluate()

results_summary = {
    "model_name": model_name,
    "model_config": SELECTED_MODEL,
    "max_length": max_length,
    "batch_size": batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "effective_batch_size": batch_size * gradient_accumulation_steps,
    "num_categories": num_categories,
    "training_time_hours": round(training_time, 2),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "eval_results": {
        "accuracy": round(eval_results["eval_accuracy"], 4),
        "f1": round(eval_results["eval_f1"], 4),
        "precision": round(eval_results["eval_precision"], 4),
        "recall": round(eval_results["eval_recall"], 4),
    },
    "timestamp": timestamp,
}

# Save results to JSON
results_file = os.path.join(output_dir, "results_summary.json")
with open(results_file, "w") as f:
    json.dump(results_summary, f, indent=2)

# Print summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE - RESULTS SUMMARY")
print("=" * 60)
print(f"Model: {model_name}")
print(f"Training Time: {training_time:.2f} hours")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall: {eval_results['eval_recall']:.4f}")
print(f"\nResults saved to: {results_file}")
print("=" * 60)

# Append results to comparison file
comparison_file = "./model_comparison.txt"
with open(comparison_file, "a") as f:
    f.write(f"\n{'=' * 60}\n")
    f.write(f"Model: {model_name}\n")
    f.write(f"Config: {SELECTED_MODEL}\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Training Time: {training_time:.2f} hours\n")
    f.write(f"Accuracy: {eval_results['eval_accuracy']:.4f}\n")
    f.write(f"F1 Score: {eval_results['eval_f1']:.4f}\n")
    f.write(f"Precision: {eval_results['eval_precision']:.4f}\n")
    f.write(f"Recall: {eval_results['eval_recall']:.4f}\n")
    f.write(f"Categories: {num_categories}\n")
    f.write(f"Train samples: {len(train_dataset)}\n")
    f.write(f"{'=' * 60}\n")

print(f"\nComparison appended to: {comparison_file}")

## 10. Model Save

---

In [ ]:
# Save the model
print("\nSaving model...")
trainer.save_model(os.path.join(output_dir, "final_model"))
print(f"Model saved to: {os.path.join(output_dir, 'final_model')}")

print("\n All done! Change SELECTED_MODEL at the top to try a different model.")
print(f"\nTo use this model for predictions:")
print(f"  from transformers import AutoTokenizer, AutoModelForSequenceClassification")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{os.path.join(output_dir, 'final_model')}')")
print(f"  model = AutoModelForSequenceClassification.from_pretrained('{os.path.join(output_dir, 'final_model')}')")

## 11. Inference

---

In [ ]:
# base = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_categories)
# Load the LoRA adapter
# model = PeftModel.from_pretrained(base, output_dir)